# 🔧 Feature Engineering — Churn de Clientes Bancários

**Objetivo:** Criar variáveis mais informativas a partir das originais, codificar categóricas e escalar numéricas — deixando o dataset pronto para a modelagem.

**Input:** `data/processed/churn_eda_completo.csv`  
**Output:** `data/processed/churn_features.csv`

---
### Estrutura deste notebook
1. Importações e carregamento
2. Criação de novas features
3. Codificação de variáveis categóricas
4. Detecção e tratamento de outliers
5. Escalonamento de variáveis numéricas
6. Seleção de features (importância preliminar)
7. Validação e exportação do dataset final

## 1. Importações e carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

COLORS = {'permaneceu': '#4C72B0', 'churn': '#DD4949'}

print('✅ Bibliotecas carregadas!')

In [ ]:
# Carregar dataset processado pela EDA
df = pd.read_csv('../data/processed/churn_eda_completo.csv')

# Remover colunas auxiliares criadas na EDA (não são features reais)
df = df.drop(columns=['churn_label', 'faixa_etaria', 'faixa_saldo'], errors='ignore')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head(3)

## 2. Criação de novas features

> Feature engineering é onde o conhecimento de negócio vira vantagem competitiva do modelo.
> Cada feature abaixo tem uma **hipótese de negócio** justificando sua criação.

In [ ]:
# ── 2.1 Features de relacionamento com o banco ────────────────────────────

# Hipótese: clientes com saldo alto mas inativos têm maior risco
df['saldo_por_produto'] = df['saldo'] / (df['num_produtos'] + 1)

# Hipótese: clientes com saldo zero são diferentes dos que têm saldo positivo
df['tem_saldo'] = (df['saldo'] > 0).astype(int)

# Hipótese: clientes com muitos anos mas poucos produtos não estão engajados
df['produtos_por_ano'] = df['num_produtos'] / (df['anos_cliente'] + 1)

# Hipótese: clientes novos (tenure baixo) podem estar em período de teste
df['cliente_novo'] = (df['anos_cliente'] <= 1).astype(int)

# Hipótese: clientes veteranos têm mais lealdade
df['cliente_veterano'] = (df['anos_cliente'] >= 8).astype(int)

print('✅ Features de relacionamento criadas:')
print('   saldo_por_produto, tem_saldo, produtos_por_ano, cliente_novo, cliente_veterano')

In [ ]:
# ── 2.2 Features de perfil financeiro ────────────────────────────────────

# Hipótese: razão saldo/salário indica comprometimento financeiro com o banco
df['razao_saldo_salario'] = df['saldo'] / (df['salario_estimado'] + 1)

# Hipótese: clientes com score de crédito baixo têm perfil de risco diferente
df['score_baixo'] = (df['score_credito'] < 500).astype(int)
df['score_alto']  = (df['score_credito'] > 750).astype(int)

# Hipótese: faixas de saldo são mais informativas que o valor bruto
df['saldo_alto'] = (df['saldo'] > df['saldo'].quantile(0.75)).astype(int)

print('✅ Features financeiras criadas:')
print('   razao_saldo_salario, score_baixo, score_alto, saldo_alto')

In [ ]:
# ── 2.3 Features de idade ─────────────────────────────────────────────────

# A EDA mostrou que idade é o preditor mais forte — explorar ao máximo

# Faixas etárias como flags binárias (mais flexíveis que uma coluna ordinal)
df['idade_18_30'] = ((df['idade'] >= 18) & (df['idade'] <= 30)).astype(int)
df['idade_31_40'] = ((df['idade'] >= 31) & (df['idade'] <= 40)).astype(int)
df['idade_41_50'] = ((df['idade'] >= 41) & (df['idade'] <= 50)).astype(int)
df['idade_51_60'] = ((df['idade'] >= 51) & (df['idade'] <= 60)).astype(int)
df['idade_60mais']= (df['idade'] > 60).astype(int)

# Transformação quadrática: captura relação não-linear com churn
df['idade_quadratica'] = df['idade'] ** 2

print('✅ Features de idade criadas:')
print('   5 flags de faixa etária + idade_quadratica')

In [ ]:
# ── 2.4 Features de interação ─────────────────────────────────────────────
# Combinações de variáveis que a EDA revelou como especialmente relevantes

# Hipótese: mulher inativa é o perfil de maior risco identificado na EDA
df['mulher_inativa'] = (
    (df['genero'] == 'Female') & (df['membro_ativo'] == 0)
).astype(int)

# Hipótese: cliente alemão com saldo alto tem risco elevado
df['alemanha_saldo_alto'] = (
    (df['pais'] == 'Germany') & (df['saldo'] > df['saldo'].quantile(0.75))
).astype(int)

# Hipótese: cliente com 4 produtos e inativo = risco máximo (achado da EDA)
df['multiprodutos_inativo'] = (
    (df['num_produtos'] >= 3) & (df['membro_ativo'] == 0)
).astype(int)

# Hipótese: idoso com saldo alto e inativo — possível migração de banco
df['idoso_rico_inativo'] = (
    (df['idade'] > 50) & (df['saldo_alto'] == 1) & (df['membro_ativo'] == 0)
).astype(int)

print('✅ Features de interação criadas:')
print('   mulher_inativa, alemanha_saldo_alto, multiprodutos_inativo, idoso_rico_inativo')

In [ ]:
# ── Verificar taxa de churn nas novas features de interação ───────────────
# Valida se as hipóteses fazem sentido nos dados

features_interacao = [
    'mulher_inativa', 'alemanha_saldo_alto',
    'multiprodutos_inativo', 'idoso_rico_inativo'
]

taxa_geral = df['churn'].mean() * 100

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, feat in enumerate(features_interacao):
    taxa = df.groupby(feat)['churn'].mean().mul(100)
    cores = ['#4C72B0', '#DD4949']
    axes[i].bar(['Não', 'Sim'], taxa.values, color=cores, width=0.4)
    axes[i].axhline(taxa_geral, color='gray', linestyle='--', linewidth=1)
    axes[i].set_title(feat.replace('_', ' ').title(), fontweight='bold', fontsize=10)
    axes[i].set_ylabel('Churn (%)')
    for j, v in enumerate(taxa.values):
        axes[i].text(j, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')
    axes[i].set_ylim(0, taxa.max() * 1.3)

plt.suptitle('Validação das Features de Interação — Taxa de Churn',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/11_validacao_features_interacao.png', bbox_inches='tight')
plt.show()

print(f'\nTaxa geral de referência: {taxa_geral:.1f}%')
for feat in features_interacao:
    taxa_sim = df[df[feat]==1]['churn'].mean() * 100
    mult = taxa_sim / taxa_geral
    print(f'{feat}: {taxa_sim:.1f}% ({mult:.1f}x a média geral)')

## 3. Codificação de variáveis categóricas

In [ ]:
# ── Estratégia de codificação ─────────────────────────────────────────────
# - genero: binária → LabelEncoder (Male=0, Female=1)
# - pais: nominal, 3 categorias → OneHot (evita ordem implícita)

# Gênero: LabelEncoder
le = LabelEncoder()
df['genero_enc'] = le.fit_transform(df['genero'])
print(f'Codificação de gênero: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# País: One-Hot Encoding
pais_dummies = pd.get_dummies(df['pais'], prefix='pais', drop_first=False)
df = pd.concat([df, pais_dummies], axis=1)
print(f'\nColunas criadas para país: {list(pais_dummies.columns)}')

# Remover originais após codificação
df = df.drop(columns=['genero', 'pais'])

print(f'\n✅ Codificação concluída. Shape: {df.shape}')

## 4. Detecção e tratamento de outliers

In [ ]:
# ── Método IQR para detectar outliers nas variáveis contínuas ─────────────
cols_outlier = ['score_credito', 'idade', 'saldo', 'salario_estimado']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

resumo_outliers = []
for i, col in enumerate(cols_outlier):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    pct = n_outliers / len(df) * 100
    resumo_outliers.append({'variavel': col, 'outliers': n_outliers, '%': round(pct, 2),
                             'lim_inf': round(lim_inf, 2), 'lim_sup': round(lim_sup, 2)})

    # Boxplot com limite IQR anotado
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#4C72B0', alpha=0.5),
                    medianprops=dict(color='#DD4949', linewidth=2))
    axes[i].set_title(f'{col}\n{n_outliers} outliers ({pct:.1f}%)', fontsize=10, fontweight='bold')
    axes[i].set_xticks([])

plt.suptitle('Detecção de Outliers — Método IQR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/12_outliers_iqr.png', bbox_inches='tight')
plt.show()

resumo_df = pd.DataFrame(resumo_outliers)
print(resumo_df.to_string(index=False))

In [ ]:
# ── Decisão: Winsorização em vez de remoção ───────────────────────────────
# Remover outliers reduziria o dataset. Winsorização preserva todos os registros
# e apenas limita os valores extremos ao percentil 1% e 99%.

cols_winsorizacao = ['score_credito', 'idade', 'saldo', 'salario_estimado']

for col in cols_winsorizacao:
    p01 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    antes = ((df[col] < p01) | (df[col] > p99)).sum()
    df[col] = df[col].clip(lower=p01, upper=p99)
    print(f'{col}: {antes} valores clippados entre [{p01:.1f}, {p99:.1f}]')

print('\n✅ Winsorização aplicada (percentis 1% e 99%)')
print('   Todos os 10.000 registros preservados.')

## 5. Escalonamento de variáveis numéricas

> Modelos baseados em distância (KNN, SVM, regressão logística) são sensíveis à escala.  
> Árvores (Random Forest, XGBoost) **não** precisam de escalonamento — mas vamos preparar as duas versões.

In [ ]:
# ── Separar features e alvo ───────────────────────────────────────────────
TARGET = 'churn'

# Colunas que NÃO entram no modelo
drop_cols = [TARGET]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[TARGET].copy()

print(f'Features: {len(feature_cols)}')
print(f'Alvo: {TARGET} — {y.sum():,} positivos ({y.mean()*100:.1f}%)')
print(f'\nLista de features:\n{feature_cols}')

In [ ]:
# ── Identificar colunas numéricas contínuas (candidatas ao escalonamento) ─
cols_escalar = [
    'score_credito', 'idade', 'anos_cliente', 'saldo',
    'salario_estimado', 'saldo_por_produto', 'razao_saldo_salario',
    'produtos_por_ano', 'idade_quadratica'
]

# Versão 1: StandardScaler (média 0, desvio 1) — para modelos lineares
X_standard = X.copy()
scaler_standard = StandardScaler()
X_standard[cols_escalar] = scaler_standard.fit_transform(X[cols_escalar])

# Versão 2: MinMaxScaler (0 a 1) — alternativa para redes neurais
X_minmax = X.copy()
scaler_minmax = MinMaxScaler()
X_minmax[cols_escalar] = scaler_minmax.fit_transform(X[cols_escalar])

# Versão 3: sem escalonamento — para árvores (Random Forest, XGBoost)
X_tree = X.copy()

print('✅ Três versões do dataset preparadas:')
print('   X_standard  → regressão logística, SVM, KNN')
print('   X_minmax    → redes neurais')
print('   X_tree      → Random Forest, XGBoost (sem escala)')

In [ ]:
# ── Visualizar efeito do escalonamento ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

cols_exemplo = ['score_credito', 'saldo', 'salario_estimado']
versoes = [
    (X, 'Original', '#4C72B0'),
    (X_standard, 'StandardScaler', '#DD8800'),
    (X_minmax, 'MinMaxScaler', '#22AA55'),
]

for col_i, col in enumerate(cols_exemplo):
    for ax_i, (df_v, titulo, cor) in enumerate(versoes):
        ax = axes[col_i // 3 * 1 + (col_i % 3 // 3), col_i % 3] if False else None

for row_i, col in enumerate(cols_exemplo[:2]):
    for col_j, (df_v, titulo, cor) in enumerate(versoes):
        ax = axes[row_i, col_j]
        sns.histplot(df_v[col], ax=ax, color=cor, bins=30, kde=True)
        media = df_v[col].mean()
        std   = df_v[col].std()
        ax.set_title(f'{col}\n{titulo}\nmédia={media:.2f}, std={std:.2f}', fontsize=9)
        ax.set_xlabel('')

plt.suptitle('Efeito do Escalonamento nas Distribuições', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/13_escalonamento.png', bbox_inches='tight')
plt.show()

## 6. Seleção de features — importância preliminar

> Usar um Random Forest rápido para ranquear quais features têm mais poder preditivo.  
> Isso orienta quais criar mais e quais descartar — raciocínio de nível pleno.

In [ ]:
# ── Random Forest rápido apenas para importância — não é o modelo final ───
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_tree, y, test_size=0.2, random_state=42, stratify=y
)

rf_prelim = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42,
    class_weight='balanced',  # já tratando o desbalanceamento
    n_jobs=-1
)
rf_prelim.fit(X_train, y_train)

print(f'Acurácia preliminar (validação): {rf_prelim.score(X_val, y_val):.4f}')
print('(Esse número será melhorado no notebook de modelagem)')

In [ ]:
# ── Importância por impureza (MDI) ────────────────────────────────────────
importancias = pd.Series(
    rf_prelim.feature_importances_,
    index=feature_cols
).sort_values(ascending=True)

# Top 20 features
top20 = importancias.tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
cores = ['#DD4949' if v > importancias.median() else '#4C72B0' for v in top20.values]
bars = ax.barh(top20.index, top20.values, color=cores, height=0.6)

ax.axvline(importancias.median(), color='gray', linestyle='--', linewidth=1)
ax.set_title('Top 20 Features por Importância (Random Forest)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importância (MDI)')

for bar, val in zip(bars, top20.values):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/14_feature_importance.png', bbox_inches='tight')
plt.show()

print('\n🔴 Features acima da mediana (vermelho) = mais importantes para o modelo')

In [ ]:
# ── Remover features de baixíssima importância ────────────────────────────
# Features com importância < 0.005 geralmente adicionam ruído ao modelo

LIMIAR_IMPORTANCIA = 0.005
features_remover = importancias[importancias < LIMIAR_IMPORTANCIA].index.tolist()
features_manter  = importancias[importancias >= LIMIAR_IMPORTANCIA].index.tolist()

print(f'Features removidas ({len(features_remover)}): {features_remover}')
print(f'Features mantidas ({len(features_manter)})')

# Aplicar em todas as versões
X_final         = X_tree[features_manter].copy()
X_final_std     = X_standard[features_manter].copy()
X_final_minmax  = X_minmax[features_manter].copy()

print(f'\nShape final do dataset: {X_final.shape}')

## 7. Validação e exportação

In [ ]:
# ── Checklist de qualidade antes de exportar ──────────────────────────────
print('='*55)
print('       CHECKLIST DE QUALIDADE — PRÉ-MODELAGEM')
print('='*55)

# 1. Valores nulos
nulos = X_final.isnull().sum().sum()
status = '✅' if nulos == 0 else '❌'
print(f'\n{status} Valores nulos: {nulos}')

# 2. Shape
print(f'✅ Shape: {X_final.shape[0]:,} linhas × {X_final.shape[1]} features')

# 3. Balanceamento da variável alvo
taxa_churn = y.mean() * 100
print(f'⚠️  Balanceamento: {taxa_churn:.1f}% churn — tratar com class_weight na modelagem')

# 4. Tipos de dados
tipos = X_final.dtypes.value_counts()
print(f'✅ Tipos: {dict(tipos)}')

# 5. Intervalo das features escaladas
std_range = X_final_std[['score_credito','saldo']].describe().loc[['min','max']]
print(f'\n✅ Escala StandardScaler (esperado: ~[-3, 3]):')
print(std_range.to_string())

mm_range = X_final_minmax[['score_credito','saldo']].describe().loc[['min','max']]
print(f'\n✅ Escala MinMaxScaler (esperado: [0, 1]):')
print(mm_range.to_string())

print('\n' + '='*55)

In [ ]:
# ── Exportar os datasets para a modelagem ────────────────────────────────

# Dataset para árvores (sem escala) — uso principal
df_export = X_final.copy()
df_export['churn'] = y.values
df_export.to_csv('../data/processed/churn_features.csv', index=False)

# Dataset com StandardScaler — para regressão logística
df_std = X_final_std.copy()
df_std['churn'] = y.values
df_std.to_csv('../data/processed/churn_features_standard.csv', index=False)

# Salvar lista de features finais
with open('../data/processed/features_selecionadas.txt', 'w') as f:
    f.write('\n'.join(features_manter))

print('✅ Arquivos exportados:')
print('   data/processed/churn_features.csv          (árvores)')
print('   data/processed/churn_features_standard.csv  (modelos lineares)')
print('   data/processed/features_selecionadas.txt    (lista de features)')
print(f'\n   Total de features finais: {len(features_manter)}')
print(f'   Total de registros: {len(df_export):,}')

## Resumo do Feature Engineering

| Categoria | Features criadas | Hipótese central |
|---|---|---|
| Relacionamento | `saldo_por_produto`, `produtos_por_ano`, `tem_saldo` | Engajamento com produtos do banco |
| Financeiro | `razao_saldo_salario`, `score_baixo`, `score_alto` | Comprometimento financeiro |
| Idade | 5 faixas + `idade_quadratica` | Relação não-linear da EDA |
| Interação | `mulher_inativa`, `alemanha_saldo_alto`, etc. | Combinações de alto risco da EDA |
| Codificação | `genero_enc`, `pais_France`, `pais_Germany`, `pais_Spain` | Modelos não leem texto |

**Total de features originais:** 9  
**Total de features após engenharia:** varia por importância (ver `features_selecionadas.txt`)

➡️ **Próximo notebook:** `03_modeling.ipynb`